In [1]:
import os
import shutil
import urllib.request, urllib.parse, urllib.error
from functools import reduce

import numpy as np
import pandas as pd

import plotly.express as px
import seaborn as sns
import matplotlib
from matplotlib import pyplot as plt

from tqdm.notebook import tqdm

In [2]:
FILTERED_SUMMARY = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_species_summary.csv'
FILTERED_METADATA = '/mnt/craig/pan_phylon/S_pyogenes/metadata/filtered_species_metadata.csv'

In [3]:
filtered_species_summary = pd.read_csv(FILTERED_SUMMARY, index_col=0, dtype='object')
filtered_species_metadata = pd.read_csv(FILTERED_METADATA, index_col=0, dtype='object')

# Ensure genome_id is a str
filtered_species_summary['genome_id'] = filtered_species_summary['genome_id'].astype('str')
filtered_species_metadata['genome_id'] = filtered_species_metadata['genome_id'].astype('str')

display(
    filtered_species_summary.shape,
    filtered_species_summary.head()
)

(372, 20)

,genome_id,genome_name,taxon_id,genome_status,genome_length,gc_content,contig_l50,contig_n50,chromosomes,plasmids,contigs,patric_cds,refseq_cds,trna,rrnacoarse_consistency,fine_consistency,checkm_completeness,checkm_contamination,genome_qualitydate_created,date_modified
817574,1314.4070,Streptococcus pyogenes 1004,1314,Complete,1842099,38.30196,1,1,1,0,1,1798,1706,67,NaN,99.9,100.0,0.0,NaN,2023-04-21T00:08:53.590Z
817575,1314.4068,Streptococcus pyogenes 1039,1314,Complete,1856557,38.30591,1,1,1,0,1,1827,1734,67,NaN,99.8,100.0,0.0,NaN,2023-04-21T00:00:24.086Z
817576,1314.4067,Streptococcus pyogenes 1042,1314,Complete,1853582,38.259544,1,1,1,0,1,1809,1719,67,NaN,99.9,100.0,0.0,NaN,2023-04-21T00:00:22.678Z
817577,1314.4071,Streptococcus pyogenes 1044,1314,Complete,2008188,38.50257,1,1,1,0,1,2090,1980,67,NaN,99.6,100.0,1.9,NaN,2023-04-21T03:00:45.142Z
817578,1314.3939,Streptococcus pyogenes 1095,1314,Complete,1854315,38.254448,1,1,1,0,1,1813,1718,67,NaN,99.9,100.0,0.0,NaN,2023-02-15T03:12:05.857Z


In [4]:
complete_strains = filtered_species_metadata.genome_status == "Complete"


display(
    filtered_species_metadata[complete_strains].shape,
    filtered_species_metadata[complete_strains].head()
)


(282, 66)

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,serovar,biovar,pathovar,mlst,...,motility,sporulation,temperature_range,optimal_temperature,salinity,oxygen_requirement,habitat,disease,comments,additional_metadata
817574,1314.4070,Streptococcus pyogenes 1004,NaN,1314,Complete,1004,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_type:Pure culture
817575,1314.4068,Streptococcus pyogenes 1039,NaN,1314,Complete,1039,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_type:Pure culture
817576,1314.4067,Streptococcus pyogenes 1042,NaN,1314,Complete,1042,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_type:Pure culture
817577,1314.4071,Streptococcus pyogenes 1044,NaN,1314,Complete,1044,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_type:Pure culture
817578,1314.3939,Streptococcus pyogenes 1095,NaN,1314,Complete,1095,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.530,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sample_type:Pure cultured organism;biomaterial...


In [5]:
download_paths = '/mnt/craig/pan_phylon/S_pyogenes/raw/genomes/'

In [6]:
# Set of genome ids to download 
#dl_strains = set(filtered_species_summary.genome_id.astype(str))
dl_strains = set(filtered_species_metadata[complete_strains].genome_id.astype(str))

# Any previously downloaded strains in directory that do not meet curent filtration criteria
# This only matters when you are re-running this notebook/workflow
bad_dl = list()

for folder in os.listdir(download_paths):
    if folder in dl_strains:
        dl_strains.remove(str(folder))        
    else:
        bad_dl.append(str(folder))

In [7]:
len(dl_strains) # if len is 0, all strains are already downloaded

282

In [8]:
len(bad_dl) # if len is 0, no bad downloads need to be removed

0

In [9]:
# delete bad_dl strains
# Only matters on a rerun
for folder in tqdm(bad_dl):
    print('removing folder:', folder)
    shutil.rmtree(os.path.join(download_paths, folder))

0it [00:00, ?it/s]

# Download (first-pass) filtered genomes

In [10]:
import subprocess
import shutil
import os
from tqdm.auto import tqdm

VALID_BV_BRC_FILES = [
    'faa', 'features.tab', 'ffn', 'frn', 'gff',
    'pathway.tab', 'spgene.tab', 'subsystem.tab', 'fna'
]

def download_bv_brc_genomes(genomes, output_dir, filetypes=['fna', 'gff'], redownload=False):
    bad_genomes = []

    if not os.path.exists(output_dir):
        os.mkdir(output_dir)

    # Process filetypes
    source_target_filetypes = []
    for ftype in tqdm(filetypes, desc='Processing filetypes...'):
        if ftype in VALID_BV_BRC_FILES:
            ftype_source = 'PATRIC.' + ftype if ftype != 'fna' else ftype
            source_target_filetypes.append((ftype_source, ftype))

        elif ftype.replace('PATRIC.', '') in VALID_BV_BRC_FILES:
            source_target_filetypes.append((ftype, ftype.replace('PATRIC.', '')))

        else:
            print('Invalid filetype:', ftype)

    # Download relevant files
    for i, genome in tqdm(
        enumerate(genomes),
        desc='Downloading selected files...',
        total=len(genomes)
    ):
        genome_dir = os.path.join(output_dir, genome)
        os.makedirs(genome_dir, exist_ok=True)

        genome_base_url = f'ftps://ftp.bvbrc.org/genomes/{genome}/{genome}'
        genome_base_local = os.path.join(genome_dir, genome)

        genome_ok = True
        for source_filetype, target_filetype in source_target_filetypes:
            source = f'{genome_base_url}.{source_filetype}'
            target = f'{genome_base_local}.{target_filetype}'

            if os.path.exists(target) and not redownload:
                print(f'{i+1} Already exists: {target}')
                continue

            print(f'{i+1} {source} -> {target}')
            result = subprocess.run([
                'wget',
                '--ftp-user=anonymous',
                '--ftp-password=guest',
                '--secure-protocol=auto',
                '--no-check-certificate',
                '--continue',
                '-q',           # quiet; remove if you want wget progress
                '-O', target,
                source,
            ])

            if result.returncode != 0:
                print(f'Bad genome ID or missing file: {genome}')
                shutil.rmtree(genome_dir, ignore_errors=True)
                bad_genomes.append(genome)
                genome_ok = False
                break  # don't try remaining filetypes for this genome

        # Clean up empty dir if something failed mid-genome
        if not genome_ok:
            continue

    return bad_genomes

In [11]:
bad_genomes = download_bv_brc_genomes(genomes=dl_strains, filetypes=['fna'], output_dir=download_paths, redownload=False)

Processing filetypes...:   0%|          | 0/1 [00:00<?, ?it/s]

1 ftps://ftp.bvbrc.org/genomes/1314.3164/1314.3164.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.3164/1314.3164.fna
2 ftps://ftp.bvbrc.org/genomes/1314.2389/1314.2389.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.2389/1314.2389.fna
3 ftps://ftp.bvbrc.org/genomes/1314.4594/1314.4594.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.4594/1314.4594.fna
4 ftps://ftp.bvbrc.org/genomes/1314.967/1314.967.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.967/1314.967.fna
5 ftps://ftp.bvbrc.org/genomes/1150773.4/1150773.4.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1150773.4/1150773.4.fna
6 ftps://ftp.bvbrc.org/genomes/1314.1019/1314.1019.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.1019/1314.1019.fna
7 ftps://ftp.bvbrc.org/genomes/1336746.3/1336746.3.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1336746.3/1336746.3.fna
8 ftps://ftp.bvbrc.org/genomes/1314.213/1314.213.fna -> /mnt/craig/pan_phylon/S_pyogenes/raw/genomes/1314.213/1314.

In [12]:
len(bad_genomes)

0

In [13]:
dl_success = sorted(dl_strains - set(bad_genomes))
len(dl_success)

282

## Remove genomes with empty fasta files

In [14]:
# some fasta files on BV-BRC are empty, remove those and remove them from metadata

## command to remove files with no text in them, list made by checking file lenth in terminal
## commented out since it has been run already

count = 0
for folder in os.listdir(download_paths):
    size = os.path.getsize(download_paths + '/' + folder + '/' + folder + '.fna' )
    if size == 0:
        count += 1
        bad_genomes.append(folder)
        print('Removing folder:', folder)
        shutil.rmtree(os.path.join(download_paths, folder))
print("Empty downloads total:", count)


Removing folder: 1314.4673
Empty downloads total: 1


# Update `summary` and `metadata` files

In [15]:
filtered_species_metadata_complete = filtered_species_metadata[complete_strains].copy()
filtered_species_metadata_wgs = filtered_species_metadata[~complete_strains].copy()

display(
    filtered_species_metadata_complete.shape,
    filtered_species_metadata_wgs.shape
)

(282, 66)

(90, 66)

In [16]:
downloaded_genomes = sorted(set(filtered_species_metadata_complete['genome_id']) - set(bad_genomes))

filtered_species_metadata_complete = (filtered_species_metadata_complete.
                                      drop_duplicates(subset=['genome_id']).
                                      set_index('genome_id').
                                      loc[downloaded_genomes].
                                      reset_index())

display(
    filtered_species_metadata_complete.shape,
    filtered_species_metadata_complete.head()
)

(281, 66)

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,serovar,biovar,pathovar,mlst,...,motility,sporulation,temperature_range,optimal_temperature,salinity,oxygen_requirement,habitat,disease,comments,additional_metadata
0,1010840.4,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,Complete,MGAS1882,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.172,...,NaN,NaN,NaN,NaN,NaN,NaN,Host,NaN,-,NaN
1,1048264.3,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,Complete,HKU QMH11M0907901,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Clinical use of next generation sequencing for...,NaN
2,1150773.3,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,serovar emm6,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,NaN,NaN,NaN,C,NaN,NaN,NaN,NaN,We report the complete genome assemblies of th...,collected_by:Rockefeller University Lancefield...
3,1150773.4,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,No,NaN,NaN,NaN,NaN,NaN,NaN,Pharyngitis,Complete genome sequence of the highly invasiv...,NaN
4,1207470.4,Streptococcus pyogenes M1 476,Streptococcus pyogenes M1 476,1207470,Complete,476,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Toxic shock syndrome,We report the completely annotated genome sequ...,NaN


In [17]:
# Common genomes list
common_genomes_complete = sorted(set(filtered_species_metadata_complete['genome_id']).intersection(set(filtered_species_summary['genome_id'])))
common_genomes_wgs = sorted(set(filtered_species_metadata_wgs['genome_id']).intersection(set(filtered_species_summary['genome_id'])))

# non-downloaded wgs genomes summary & metadata
filtered_species_metadata_wgs = filtered_species_metadata.set_index('genome_id').loc[common_genomes_wgs].reset_index()
filtered_species_summary_wgs = filtered_species_summary.set_index('genome_id').loc[common_genomes_wgs].reset_index()

# downloaded complete genomes summary & metadata
filtered_species_metadata = filtered_species_metadata_complete.set_index('genome_id').loc[common_genomes_complete].reset_index()
filtered_species_summary = filtered_species_summary.set_index('genome_id').loc[common_genomes_complete].reset_index()

display(
    filtered_species_summary.shape,
    filtered_species_summary.head(),
    filtered_species_metadata.shape,
    filtered_species_metadata.head(),
    'non-dowloaded summary & metadata',
    filtered_species_summary_wgs.shape,
    filtered_species_metadata_wgs.shape,
)

(281, 20)

,genome_id,genome_name,taxon_id,genome_status,genome_length,gc_content,contig_l50,contig_n50,chromosomes,plasmids,contigs,patric_cds,refseq_cds,trna,rrnacoarse_consistency,fine_consistency,checkm_completeness,checkm_contamination,genome_qualitydate_created,date_modified
0,1010840.4,Streptococcus pyogenes MGAS1882,1010840,Complete,1781029,38.0,1,1,1,0,1,1727,0,57,NaN,99.5,100.0,0.0,NaN,2015-03-16T03:17:09.594Z
1,1048264.3,Streptococcus pyogenes HKU QMH11M0907901,1048264,Complete,1908100,38.45,1,1,1,NaN,1,1909,1865,67,NaN,99.9,100.0,0.9,NaN,2016-01-17T15:29:01.552Z
2,1150773.3,Streptococcus pyogenes JRS4,1150773,Complete,1811968,38.63,1,1,1,NaN,1,1811,1671,67,NaN,99.8,100.0,0.0,NaN,2016-01-17T16:03:54.402Z
3,1150773.4,Streptococcus pyogenes JRS4,1150773,Complete,1811124,38.64,1,1,1,NaN,1,1886,1890,66,NaN,99.9,100.0,0.0,NaN,2016-03-01T06:31:23.641Z
4,1207470.4,Streptococcus pyogenes M1 476,1207470,Complete,1831079,38.5,1,1,1,0,1,1929,1849,57,NaN,97.9,100.0,5.0,NaN,2015-03-16T03:17:09.594Z


(281, 66)

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,serovar,biovar,pathovar,mlst,...,motility,sporulation,temperature_range,optimal_temperature,salinity,oxygen_requirement,habitat,disease,comments,additional_metadata
0,1010840.4,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,Complete,MGAS1882,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.172,...,NaN,NaN,NaN,NaN,NaN,NaN,Host,NaN,-,NaN
1,1048264.3,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,Complete,HKU QMH11M0907901,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Clinical use of next generation sequencing for...,NaN
2,1150773.3,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,serovar emm6,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,NaN,NaN,NaN,C,NaN,NaN,NaN,NaN,We report the complete genome assemblies of th...,collected_by:Rockefeller University Lancefield...
3,1150773.4,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,No,NaN,NaN,NaN,NaN,NaN,NaN,Pharyngitis,Complete genome sequence of the highly invasiv...,NaN
4,1207470.4,Streptococcus pyogenes M1 476,Streptococcus pyogenes M1 476,1207470,Complete,476,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Toxic shock syndrome,We report the completely annotated genome sequ...,NaN


'non-dowloaded summary & metadata'

(90, 20)

(90, 66)

# Ensure complete sequences have parity with contig count

Complete sequences should not have more than 12 contigs total (1 main chromosome + 11 plasmids)

In [18]:
(filtered_species_metadata.contigs.astype(int) > 11).sum() # number of strains with too many contigs

0

In [19]:
# These strains need to be investigated
problem_strains = filtered_species_metadata.contigs.astype(int) > 11
filtered_species_metadata[problem_strains]

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,serovar,biovar,pathovar,mlst,...,motility,sporulation,temperature_range,optimal_temperature,salinity,oxygen_requirement,habitat,disease,comments,additional_metadata


In [20]:
# BioProject numbers don't line up with number of contigs, need to be removed
filtered_species_metadata[problem_strains][['genome_id', 'bioproject_accession', 'contigs']]

,genome_id,bioproject_accession,contigs


In [21]:
filtered_species_metadata = filtered_species_metadata.set_index('genome_id').drop(labels=filtered_species_metadata[problem_strains].genome_id).reset_index()
filtered_species_summary = filtered_species_summary.set_index('genome_id').loc[filtered_species_metadata.genome_id.values].reset_index()

display(
    filtered_species_summary.shape,
    filtered_species_summary.head(),
    filtered_species_metadata.shape,
    filtered_species_metadata.head(),
)

(281, 20)

,genome_id,genome_name,taxon_id,genome_status,genome_length,gc_content,contig_l50,contig_n50,chromosomes,plasmids,contigs,patric_cds,refseq_cds,trna,rrnacoarse_consistency,fine_consistency,checkm_completeness,checkm_contamination,genome_qualitydate_created,date_modified
0,1010840.4,Streptococcus pyogenes MGAS1882,1010840,Complete,1781029,38.0,1,1,1,0,1,1727,0,57,NaN,99.5,100.0,0.0,NaN,2015-03-16T03:17:09.594Z
1,1048264.3,Streptococcus pyogenes HKU QMH11M0907901,1048264,Complete,1908100,38.45,1,1,1,NaN,1,1909,1865,67,NaN,99.9,100.0,0.9,NaN,2016-01-17T15:29:01.552Z
2,1150773.3,Streptococcus pyogenes JRS4,1150773,Complete,1811968,38.63,1,1,1,NaN,1,1811,1671,67,NaN,99.8,100.0,0.0,NaN,2016-01-17T16:03:54.402Z
3,1150773.4,Streptococcus pyogenes JRS4,1150773,Complete,1811124,38.64,1,1,1,NaN,1,1886,1890,66,NaN,99.9,100.0,0.0,NaN,2016-03-01T06:31:23.641Z
4,1207470.4,Streptococcus pyogenes M1 476,1207470,Complete,1831079,38.5,1,1,1,0,1,1929,1849,57,NaN,97.9,100.0,5.0,NaN,2015-03-16T03:17:09.594Z


(281, 66)

,genome_id,genome_name,organism_name,taxon_id,genome_status,strain,serovar,biovar,pathovar,mlst,...,motility,sporulation,temperature_range,optimal_temperature,salinity,oxygen_requirement,habitat,disease,comments,additional_metadata
0,1010840.4,Streptococcus pyogenes MGAS1882,Streptococcus pyogenes MGAS1882,1010840,Complete,MGAS1882,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.172,...,NaN,NaN,NaN,NaN,NaN,NaN,Host,NaN,-,NaN
1,1048264.3,Streptococcus pyogenes HKU QMH11M0907901,NaN,1048264,Complete,HKU QMH11M0907901,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.36,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Clinical use of next generation sequencing for...,NaN
2,1150773.3,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,serovar emm6,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,NaN,NaN,NaN,C,NaN,NaN,NaN,NaN,We report the complete genome assemblies of th...,collected_by:Rockefeller University Lancefield...
3,1150773.4,Streptococcus pyogenes JRS4,NaN,1150773,Complete,JRS4,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.37,...,No,NaN,NaN,NaN,NaN,NaN,NaN,Pharyngitis,Complete genome sequence of the highly invasiv...,NaN
4,1207470.4,Streptococcus pyogenes M1 476,Streptococcus pyogenes M1 476,1207470,Complete,476,NaN,NaN,NaN,MLST.Streptococcus_pyogenes.28,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Toxic shock syndrome,We report the completely annotated genome sequ...,NaN


In [22]:
(filtered_species_metadata['genome_id'] == '1314.739').sum() # ensure reference strain is still part of the filtered & downloaded dataset

1

In [23]:
# Filtered & downloaded summary
filepath = os.path.join(FILTERED_SUMMARY.split('filtered_species_summary.csv')[0], 'filtered_downloaded_species_summary.csv')
filtered_species_summary.to_csv(filepath)

In [24]:
# Filtered & downloaded metadata
filepath = os.path.join(FILTERED_METADATA.split('filtered_species_metadata.csv')[0], 'filtered_downloaded_species_metadata.csv')
filtered_species_metadata.to_csv(filepath)

In [25]:
# Filter & NOT downloaded summary & metadata
filepath = os.path.join(FILTERED_SUMMARY.split('filtered_species_summary.csv')[0], 'filtered_NOT_downloaded_species_summary.csv')
filtered_species_summary_wgs.to_csv(filepath)

filepath = os.path.join(FILTERED_METADATA.split('filtered_species_metadata.csv')[0], 'filtered_NOT_downloaded_species_metadata.csv')
filtered_species_metadata_wgs.to_csv(filepath)